Basic settings

In [5]:
from utgc.runner import EnsembleRunner
runner = EnsembleRunner.from_config('output/SA/testrun/config.yaml')

Initializing runner from configuration: output/SA/testrun/config.yaml
Loaded 2745 segments from data/UT_vtds.geojson
Loaded 4 districts from maps/US-House/2025_USH_Leg-C/2025_USH_Leg-C.shp
Projecting to EPSG:26912
Found 1978 nodes assigned to 254 incorporated municipalities
Assigning unique IDs to unincorporated nodes...
Assigned unique IDs to 767 unincorporated nodes
Total unique MUNIIDs: 1021
  Memory optimization: Removed geometries from graph nodes
  Geodata preserved for updaters requiring geometric information
  Graph built with 2745 nodes, 7641 edges
Population deviation tolerance: 0.100000%
Constraint: split max 4 muni
Constraint: max 0 multi-splits of muni
  Added locality split updater: 'ls_muni'
    Ignoring updater in output: 'ls_muni'
  Added split updater: 'split_muni'
  Added multi-split updater: 'muni_multi_splits'
Constraint: split max 5 county
Constraint: max 0 multi-splits of county
  Added locality split updater: 'ls_county'
    Ignoring updater in output: 'ls_count

Election settings

In [6]:
runner = (runner
    .add_election_updater(
        name="2016PRE",
        parties_to_columns={"D":"G16PREDCLI","R":"G16PRERTRU","-":"G16PRE-TOT"}
    )
    .add_election_updater(
        name="2016GOV",
        parties_to_columns={"D":"G16GOVDWEI","R":"G16GOVRHER","-":"G16GOV-TOT"}
    )
    .add_election_updater(
        name="2016ATG",
        parties_to_columns={"D":"G16ATGDHAR","R":"G16ATGRREY","-":"G16ATG-TOT"}
    )
    .add_election_updater(
        name="2016AUD",
        parties_to_columns={"D":"G16AUDDMIT","R":"G16AUDRDOU","-":"G16AUD-TOT"}
    )
    .add_election_updater(
        name="2016TRE",
        parties_to_columns={"D":"G16TREDHAN","R":"G16TRERDAM","-":"G16TRE-TOT"}
    )
    .add_election_updater(
        name="2020PRE",
        parties_to_columns={"D":"G20PREDBID","R":"G20PRERTRU","-":"G20PRE-TOT"}
    )
    .add_election_updater(
        name="2020GOV",
        parties_to_columns={"D":"G20GOVDPET","R":"G20GOVRCOX","-":"G20GOV-TOT"}
    )
    .add_election_updater(
        name="2020ATG",
        parties_to_columns={"D":"G20ATGDSKO","R":"G20ATGRREY","-":"G20ATG-TOT"}
    )
    .add_election_updater(
        name="2024PRE",
        parties_to_columns={"D":"G24PREDHAR","R":"G24PRERTRU","-":"G24PRE-TOT"}
    )
    .add_election_updater(
        name="2024GOV",
        parties_to_columns={"D":"G24GOVDCUM","R1":"G24GOVRHEN","R2":"G24GOVNCLA","-":"G24GOV-TOT"}
    )
    .add_election_updater(
        name="2024ATG",
        parties_to_columns={"D":"G24ATGDBAU","R":"G24ATGRBRO","-":"G24ATG-TOT"}
    )
    .add_election_updater(
        name="2024AUD",
        parties_to_columns={"D":"G24AUDDVOU","R":"G24AUDRCAN","-":"G24AUD-TOT"}
    )
    .add_election_updater(
        name="2024TRE",
        parties_to_columns={"D":"G24TREDHAN","R":"G24TREROAK","-":"G24TRE-TOT"}  
    )
    .add_election_aggregator(
        "sb1011_data",
        ["2016PRE", "2016GOV", "2016ATG", "2016AUD", "2016TRE", "2020PRE", "2020GOV", "2020ATG", "2024PRE", "2024GOV", "2024ATG", "2024AUD", "2024TRE"],
        parties=["D", "R", "-"]
    )
    .add_election_metric_updaters(
        "sb1011_data",
        ["partisan_bias_utah", "partisan_bias", "mean_median", "efficiency_gap", "stdev_partisan_share", "majority_partisan_shares", "majority_seats"]
    )
)

  Added election updater: '2016PRE'
    Ignoring updater in output: '2016PRE'
  Added election updater: '2016GOV'
    Ignoring updater in output: '2016GOV'
  Added election updater: '2016ATG'
    Ignoring updater in output: '2016ATG'
  Added election updater: '2016AUD'
    Ignoring updater in output: '2016AUD'
  Added election updater: '2016TRE'
    Ignoring updater in output: '2016TRE'
  Added election updater: '2020PRE'
    Ignoring updater in output: '2020PRE'
  Added election updater: '2020GOV'
    Ignoring updater in output: '2020GOV'
  Added election updater: '2020ATG'
    Ignoring updater in output: '2020ATG'
  Added election updater: '2024PRE'
    Ignoring updater in output: '2024PRE'
  Added election updater: '2024GOV'
    Ignoring updater in output: '2024GOV'
  Added election updater: '2024ATG'
    Ignoring updater in output: '2024ATG'
  Added election updater: '2024AUD'
    Ignoring updater in output: '2024AUD'
  Added election updater: '2024TRE'
    Ignoring updater in outp

In [7]:
# Set up visualization of sample maps
import os
import utgc.plotting as gcplot
import utgc.notebookhelper as nbh

# Generate municipality and county boundaries from the geodata
munis, counties = nbh.generate_boundaries_from_geodata(runner.geodata)

runner = runner.add_runtime_callback(
    name="render_maps",
    frequency=10,
    action=lambda partition, step, output_dir: gcplot.visualize_partition(
        partition,
        step,
        os.path.join(output_dir, "maps"),
        counties=counties,
        municipalities=munis,
        split_munis_count=partition["split_muni"],
        split_counties_count=partition["split_county"],
    )
)

Registered callback 'render_maps' (<lambda>) to run every 10 steps.


In [ ]:
runner = runner.precondition(steps=50)
runner.run(
    name="ensemble",
    output_dir="output/SA/",
    num_steps=250,
)

=== Preconditioning ===
Starting preconditioning...


 12%|█▏        | 6/50 [00:06<00:45,  1.04s/it]


KeyboardInterrupt: 

In [ ]:
from utgc.results import ResultSet
import matplotlib.pyplot as plt
results = ResultSet(output_file="output/SA/ensemble/output.jsonl")

results.plot_metric_violin("majority_partisan_shares")